# BrainPrint Brainwave: run TRIBE v2 on the panel questions

Meta's **TRIBE v2** predicts how an *average* brain responds, second by second, to a text. BrainPrint shows that response next to what each persona brain thinks of the same point.

1. **Runtime → Change runtime type → T4 GPU.**
2. Your Hugging Face account needs access to `meta-llama/Llama-3.2-3B` (TRIBE's text encoder).
3. Run the cells in order: the install cell, then the torchaudio fix cell, then **Runtime → Restart session** once, then continue from step 2. Do not run the install cells again after the restart.
4. At the end `brainwave_results.zip` downloads. Unzip the JSON files into `brainwave/results/` in your project folder, then click **Import TRIBE results** on the Brain Panel page.

The inputs are the questions and pitches exported with `node brainwave/export.mjs` (in `brainwave/inputs/`). Each one takes about 3 to 5 minutes on a T4.

**What the numbers are:** predicted fMRI-like responses for an average subject, grouped into 7 coarse networks. They are not measurements of any real person, and not of any persona brain.

In [ ]:
# Step 1a: install
!pip install -q "tribev2 @ git+https://github.com/facebookresearch/tribev2.git" nilearn==0.11.1

In [ ]:
# Step 1b: make torchaudio match the torch version tribev2 installed (avoids "undefined symbol" / "_init_sox" errors)
import torch, subprocess, sys
ver = torch.__version__.split("+")[0]
cuda = torch.version.cuda.replace(".", "")
print("torch", ver, "cuda", cuda)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", f"torchaudio=={ver}",
                "--index-url", f"https://download.pytorch.org/whl/cu{cuda}"], check=True)
print("ok: torchaudio", ver, "- now do Runtime -> Restart session, then continue from step 2")

## Step 2: log in to Hugging Face and fetch BrainPrint

In [ ]:
from huggingface_hub import login
login()  # paste a READ token from the account that has Llama-3.2-3B access. Never share it.

!git clone -q https://github.com/shubham123sing/brainprint.git 2>/dev/null || (cd brainprint && git pull -q)
import sys, json, os
sys.path.insert(0, "brainprint/legacy/tribe")
manifest = json.load(open("brainprint/brainwave/inputs/manifest.json"))
for m in manifest: print(m["id"], "|", m["title"], "|", m["words"], "words")

## Step 3: run TRIBE v2 on every input (skips the ones already done)

In [ ]:
import run_tribe
from networks import load_atlas

model = run_tribe.load_model("./tribe_cache")   # downloads about 1 GB of weights the first time
atlas = load_atlas()
os.makedirs("brainwave_results", exist_ok=True)
for m in manifest:
    out = f"brainwave_results/{m['id']}.json"
    if os.path.exists(out):
        print("done already:", m["id"]); continue
    run_tribe.run(model, f"brainprint/brainwave/inputs/{m['id']}.txt", m["id"], m["title"], out, atlas=atlas)

## Step 4: download

In [ ]:
!cd brainwave_results && zip -q ../brainwave_results.zip *.json
from google.colab import files
files.download("brainwave_results.zip")

## Optional: plot the 7 networks

In [ ]:
import matplotlib.pyplot as plt
for m in manifest:
    d = json.load(open(f"brainwave_results/{m['id']}.json"))
    t = [s["t"] for s in d["seconds"]]
    plt.figure(figsize=(10, 2.5)); plt.title(f"{m['title']}: predicted network response (TRIBE v2, average subject)")
    for n in d["networks"]:
        plt.plot(t, [s["networks"][n] for s in d["seconds"]], label=n)
    plt.legend(fontsize=7, ncol=7); plt.xlabel("seconds"); plt.show()